In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("Test Conncetion")
    .config("spark.jars", "postgresql-42.7.3.jar")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "128g")
    .config("spark.executor.memory", "128g")
    .getOrCreate()
)


url = "jdbc:postgresql://127.0.0.1:5432/wikitest"
properties = {"user": "richard", "password": "rich", "driver": "org.postgresql.Driver"}

pages_df = spark.read.jdbc(url, "pages", properties=properties)
users_df = spark.read.jdbc(url, "users", properties=properties)
revisions_df = spark.read.jdbc(url, "revisions", properties=properties)

24/04/03 19:26:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [8]:
from pyspark.sql.functions import col, substring, max

# Convert the timestamp to a date string for comparison
revisions_df = revisions_df.withColumn("date", substring("timestamp", 1, 8))

# Filter the pages DataFrame based on the namespace and deleted status
filtered_pages_df = pages_df.filter((col("namespace") == 1) & (col("deleted") == False))

# Filter the revisions DataFrame based on the date and deleted status
filtered_revisions_df = revisions_df.filter(
    (col("date") <= "20230101")
    & (col("deleted_text") == False)
    & (col("deleted_comment") == False)
    & (col("deleted_user") == False)
).withColumnRenamed("id", "revision_id")

# Join the filtered pages and revisions DataFrames
joined_df = filtered_pages_df.join(
    filtered_revisions_df,
    filtered_pages_df["id"] == filtered_revisions_df["page_id"],
)

# Find the latest revision for each page
latest_revisions_df = joined_df.groupBy("page_id").agg(
    max("timestamp").alias("latest_timestamp")
)

# Find the latest revision for each page
latest_revisions_df = joined_df.groupBy("page_id").agg(
    max("timestamp").alias("latest_timestamp")
)

# Aliasing the DataFrames for clearer reference
latest_revisions_df_alias = latest_revisions_df.alias("lr")
joined_df_alias = joined_df.alias("jd")

# Join the latest revisions with the joined DataFrame and select the desired columns using aliases
result_df = latest_revisions_df_alias.join(
    joined_df_alias,
    (col("lr.page_id") == col("jd.page_id"))
    & (col("lr.latest_timestamp") == col("jd.timestamp")),
).select(
    col("jd.page_id"),
    col("jd.title"),
    col("jd.revision_id"),
    col("jd.timestamp"),
    col("jd.user_id"),
    col("jd.minor"),
    col("jd.comment"),
    col("jd.text"),
    col("jd.bytes"),
    col("jd.sha1"),
    col("jd.model"),
    col("jd.format"),
)

In [9]:
example_commit = result_df.select("text").first()["text"]

In [10]:
print(example_commit)

{{Skip to talk}}
{{Talk header|bot=Lowercase sigmabot III|age=60}}
{{WikiProject banner shell|class=b|collapsed=yes|1=
{{Vital article|topic=Art|level=5}}
{{WikiProject Objectivism|importance=top}}
{{WikiProject Novels|importance=high}}
{{WikiProject Philosophy|importance=Mid|literature=yes|contemporary=yes|social-and-political=yes}}
{{WikiProject Politics|importance=Low|libertarianism=yes|libertarianism-importance=high}}
{{WikiProject Trains|importance=low}}
{{WikiProject Women writers|importance=High}}
}}
{{top 25 report|October 13, 2013}}
{{User:MiszaBot/config
|archiveheader = {{aan}}
|maxarchivesize = 100K
|counter = 4
|minthreadsleft = 5
|algo = old(60d)
|archive = Talk:Atlas Shrugged/Archive %(counter)d
}}

== External links modified ==

Hello fellow Wikipedians,

I have just modified {{plural:1|one external link|1 external links}} on [[Atlas Shrugged]]. Please take a moment to review [https://en.wikipedia.org/w/index.php?diff=prev&oldid=745408412 my edit]. If you have any quest

In [16]:
import re
import mwparserfromhell as mwp


def parse_revision(revision_text):
    # Regular expressions to match the desired information
    user_regex = re.compile(r"\[\[User:(.+?)\|")
    date_regex = re.compile(r"(\d{2}:\d{2}, \d{1,2} \w+ \d{4}) \(UTC\)")
    section_regex = re.compile(r"^\=\=\s*(.+?)\s*\=\=", re.MULTILINE)

    # Split the revision text into lines
    lines = revision_text.split("\n")

    # Initialize variables to store the extracted information
    messages = []
    section = None
    message_id = 0
    message_content = ""

    # Iterate over each line in the revision text
    for line in lines:
        # Check if the line contains a section heading
        section_match = section_regex.match(line)
        if section_match:
            section = section_match.group(1)
        message_content += line + "\n"
        # Check if the line contains a user message
        user_match = user_regex.search(line)
        date_match = date_regex.search(line)

        # Save the previous message and reset the message content
        if user_match and date_match:
            message_id += 1
            user = user_match.group(1)
            date = date_match.group(1)
            indentation = len(line) - len(line.lstrip(":"))
            reply_to = None
            # Determine the message being replied to based on indentation
            if indentation > 0:
                for prev_message in reversed(messages):
                    if prev_message["indentation"] < indentation:
                        reply_to = prev_message["id"]
                        break

            parsed_content = mwp.parse(message_content)
            message_plain = parsed_content.strip_code().strip()
            messages.append(
                {
                    "id": message_id,
                    "section": section,
                    "user": user,
                    "date": date,
                    "content": message_plain,
                    "reply_to": reply_to,
                    "indentation": indentation,
                }
            )
            message_content = ""
    return messages


parsed_revision = pd.DataFrame(parse_revision(example_commit))

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
import json

keywords = []
with open("keywords.jsonl", "r") as f:
    for line in f:
        keyword = json.loads(line)["word"]
        keywords.append(keyword)

keyword_lemmas = [i.lemma_ for i in nlp(" ".join(keywords))]


def check_keywords(sentence):
    # Process the sentence
    doc = nlp(sentence)
    # Check if each keyword is present in the sentence
    found_keywords = []
    found = False
    for token in doc:
        if token.lemma_ in keyword_lemmas:
            found_keywords.append(token.text)
            found = True

    return found_keywords, found


found_keywords, found = check_keywords(parsed_revision[0]["content"])
print("Found keywords:", found_keywords)
print("Keywords found:", found)

In [18]:
from detoxify import Detoxify

model = Detoxify("original", device="cuda")
# each model takes in either a string or a list of strings

results = model.predict(parsed_revision["content"].to_list())

results

{'toxicity': [0.0005713382852263749,
  0.0005700510228052735,
  0.0006119944155216217,
  0.003988251090049744,
  0.0005954811349511147,
  0.0006421711877919734,
  0.0005613457760773599],
 'severe_toxicity': [0.00012612015416380018,
  0.00012445372703950852,
  0.00012252773740328848,
  0.00010188736632699147,
  0.0001241610589204356,
  0.0001224277657456696,
  0.0001287114864680916],
 'obscene': [0.00018084469775203615,
  0.00018151896074414253,
  0.00017834083701018244,
  0.0003302147670183331,
  0.0001811670808820054,
  0.00019246958254370838,
  0.0001829825050663203],
 'threat': [0.00012605714437086135,
  0.0001210550544783473,
  0.00012524062185548246,
  0.00013591666356660426,
  0.00012467477063182741,
  0.00012087891809642315,
  0.00012847912148572505],
 'insult': [0.00017533532809466124,
  0.00017825141549110413,
  0.00017935430514626205,
  0.0003184227389283478,
  0.00018017929687630385,
  0.00017423105600755662,
  0.0001763555919751525],
 'identity_attack': [0.00014298214227892